In [ ]:
!pip install langchain langchain-openai langchain-core langgraph

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_ticket_price(route: str) -> str:
    """Get the ticket price for a given flight route."""
    prices = {
        "JFK to LAX": "$320",
        "BUF to MIA": "$280",
        "JFK to LHR": "$740",
    }
    route = route.upper().strip()
    for key, value in prices.items():
        if all(code in route for code in key.split(" to ")):
            return value
    return "No price found."

@tool
def get_route_demand(route: str) -> str:
    """Get the demand level for a given flight route."""
    demand = {
        "JFK to LAX": "High demand",
        "BUF to MIA": "Medium demand",
        "JFK to LHR": "Very high demand",
    }
    route = route.upper().strip()
    for key, value in demand.items():
        if all(code in route for code in key.split(" to ")):
            return value
    return "Demand data unavailable."

llm = ChatOpenAI(model="gpt-4o-mini")
memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_ticket_price, get_route_demand],
    system_prompt="You are an airline analytics assistant. Use tools to answer questions about airfare and demand. You also love telling airline-related jokes whenever you get the chance.",
    checkpointer=memory
)

config = {"configurable": {"thread_id": "demo-session"}}

def chat(user_input):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

print("Agent ready!")

Agent ready!


In [ ]:
chat(" ")

Agent: You have $300 for your vacation. 

Just remember to always leave some room in your budget for souvenirs, snacks, or maybe even a new travel buddy! Or should I say, "plane" friend? 😄✈️

